# 🚀 AGAR-RL V3 : Pipeline SOTA Deep RL (MultiDiscrete & Minimalist Rewards)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Albin0903/agario/blob/main/notebooks/train_colab.ipynb)

**Entraînement de haute performance à pleine puissance (2h à 3h sur GPU L4 / TPU) avec auto-sauvegarde Google Drive.**

### 🎯 Nouveautés Majeures de la V3 SOTA :
1. **Espace d'Action MultiDiscrete([24, 3])** : 24 angles directionnels précis + 3 modes nets `[0: Mouvement, 1: Split, 2: Éjecter]`. Résout définitivement la paralysie et l'absence de split en mode déterministe.
2. **Récompense Minimaliste Pure (Standard AgarCL & AgarIA)** : Récompense strictement alignée sur le delta de masse $\Delta M / M_0$ et les kills (+10), suppression intégrale des 14 micro-pénalités artificielles et du bonus passif de survie.
3. **Haute Densité de Jeu** : Arène 1400x1400 avec 20 bots adverses et 2000 pellets de nourriture (avec grappes autour des virus).
4. **Export ONNX & Visualisation HD** : Pipeline d'export immédiat et enregistrement vidéo HD avec overlay directionnel et radar minimap.


## 0. Montage Google Drive & Détection GPU L4
Tous les checkpoints, les replays HD et les modèles ONNX seront automatiquement sauvegardés sur votre Drive dans le dossier `agario_rl_backup_v3`.


In [ ]:
# 1. Montage sécurisé de Google Drive
import os, sys, time, torch

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except ImportError:
    pass

DRIVE_BACKUP_DIR = '/content/drive/MyDrive/agario_rl_backup_v3'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)

# 2. Vérification du matériel accéléré (GPU L4 / A100 recommandé)
print('=' * 65)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'🚀 Accélération GPU Détectée : {gpu_name} ({vram:.1f} Go VRAM)')
    print('⚡ Configuration optimale : 24 environnements parallèles + Numba JIT + Batch 1024')
else:
    print('⚠️ Aucun GPU détecté. Activez un GPU dans : Exécution > Modifier le type d\'exécution')
print(f'📁 Dossier Google Drive synchronisé : {DRIVE_BACKUP_DIR}')
print('=' * 65)


## 1. Synchronisation du Code GitHub & Installation des Dépendances

In [ ]:
import os

# 1. Récupération propre des dernières modifications ou clonage
if os.path.exists('.git'):
    print('🔄 Synchronisation avec GitHub main...')
    !git fetch origin main
    !git reset --hard origin/main
elif os.path.exists('agario/.git'):
    print('🔄 Déplacement dans agario et synchronisation avec GitHub main...')
    %cd agario
    !git fetch origin main
    !git reset --hard origin/main
else:
    print('🌐 Clonage propre du repo...')
    !git clone https://github.com/Albin0903/agario.git
    %cd agario

# 2. Configuration du PYTHONPATH et installation des dépendances Farama Gymnasium
os.environ['PYTHONPATH'] = f"{os.getcwd()}:{os.environ.get('PYTHONPATH', '')}"
!pip uninstall -y -q gym 2>/dev/null || true
!pip install -q -r requirements.txt tensorboard
!apt-get install -qq -y ffmpeg
print('✅ Environnement et dépendances installés avec succès.')


## 2. Validation Pré-Vol : Suite de 27 Tests Unitaires
Vérification complète de la physique Ogar, de la conformité Farama Gymnasium, des pénalités anti-corner, et de l'export ONNX.

In [ ]:
# Exécute tous les tests du moteur physique Ogar, des règles Gymnasium et de l'export ONNX
!python -m pytest -v


## 3. Monitoring TensorBoard (Optionnel)

In [ ]:
import os
os.makedirs('logs/tensorboard', exist_ok=True)
try:
    %load_ext tensorboard
    %tensorboard --logdir logs/tensorboard
except Exception as e:
    print(f'Note TensorBoard : {e}')


## 4. Entraînement Haute Performance V3 (15 000 000 Steps ≈ 2h15 sur GPU L4)
- **Architecture** : MLP 2x512, batch 512, 16 environnements parallèles (SubprocVecEnv).
- **Règles V3 SOTA** : Actions MultiDiscrete([24, 3]), récompense Delta-Masse pure, 20 bots actifs, pool de self-play périodique.
- **Sauvegarde continue** : Checkpoints automatiques tous les 250 000 pas dans `agario_rl_backup_v3`.


In [ ]:
# 🚀 Lancement à pleine puissance sur GPU NVIDIA L4 (Architecture V3)
!python src/training/train_colab.py \
    --n-envs 16 \
    --total-timesteps 15000000 \
    --pool-interval 250000 \
    --backup-dir /content/drive/MyDrive/agario_rl_backup_v3 \
    --resume none \
    --device auto


## 5. Enregistrement Automatique du Match Replay HD & Visualisation Directe
Génère une vidéo HD de 80 secondes (2400 steps @ 30 FPS) avec affichage tête haute (HUD), vecteurs de décision et radar.

In [ ]:
import os, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

# 1. Recherche prioritaire du modèle 15M (Drive ou local)
candidates = [
    '/content/drive/MyDrive/agario_rl_backup_v2/ppo_step_15000000.zip',
    '/content/drive/MyDrive/agario_rl_backup_v2/ppo_final.zip',
    'checkpoints/ppo/ppo_final.zip',
    '/content/drive/MyDrive/agario_rl_backup_v2/ppo_latest.zip',
    'checkpoints/ppo/ppo_latest.zip',
]
target_model = next((c for c in candidates if os.path.exists(c)), None)

if not target_model:
    all_ckpts = glob.glob('/content/drive/MyDrive/agario_rl_backup_v2/*.zip') + glob.glob('checkpoints/self_play_pool/*.zip')
    if all_ckpts:
        all_ckpts.sort(key=extract_step)
        target_model = all_ckpts[-1]
    else:
        target_model = 'checkpoints/ppo/ppo_latest.zip'

step_count = extract_step(target_model)
print('=' * 75)
print(f'🎬 Modèle sélectionné pour l\'évaluation HD : {target_model}')
print(f'📊 Palier de pas : {step_count:,} steps')
print('=' * 75)

# 2. Enregistrement du replay avec lissage et masquage d\'action (2400 steps @ 30 FPS = 80s)
os.makedirs('recordings', exist_ok=True)
!python src/inference/record_match.py \
    --model "{target_model}" \
    --output recordings/eval_match_v2.mp4 \
    --steps 2400

# 3. Sauvegarde sur Google Drive
if os.path.exists('recordings/eval_match_v2.mp4') and os.path.exists('/content/drive/MyDrive/agario_rl_backup_v2'):
    !cp recordings/eval_match_v2.mp4 /content/drive/MyDrive/agario_rl_backup_v2/eval_match_v2.mp4
    print('📁 Replay HD copié sur Google Drive dans : agario_rl_backup_v2/eval_match_v2.mp4')

# 4. Visualisation directe dans le Notebook
video_path = 'recordings/eval_match_v2.mp4'
if os.path.exists(video_path):
    mp4_bytes = open(video_path, 'rb').read()
    data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
    display(HTML(f'''
    <video width="850" height="480" controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
    print(f'Taille de la vidéo : {os.path.getsize(video_path) / 1_000_000:.1f} Mo')
else:
    print('⚠️ Vidéo non trouvée.')


## 6. Exportation Universelle vers ONNX & Benchmark de Latence
Convertit le réseau de neurones PyTorch au standard ONNX ultra-rapide (< 0.02 ms de latence CPU).

In [ ]:
import os, glob, re

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

candidates = glob.glob('/content/drive/MyDrive/agario_rl_backup_v3/*.zip') + \
             glob.glob('/content/drive/MyDrive/agario_rl_backup_v2/*.zip') + \
             glob.glob('checkpoints/ppo/*.zip') + \
             glob.glob('checkpoints/self_play_pool/*.zip')

candidates = sorted(list(set(candidates)), key=extract_step, reverse=True)
best_model = candidates[0] if candidates else None

if best_model and os.path.exists(best_model):
    print(f'Modèle sélectionné pour l\'export : {best_model}')
    os.makedirs('models', exist_ok=True)
    !python src/inference/export_onnx.py --model "{best_model}" --output models/model_v3.onnx
    if os.path.exists('/content/drive/MyDrive/agario_rl_backup_v3'):
        !cp models/model_v3.onnx /content/drive/MyDrive/agario_rl_backup_v3/model_v3.onnx
        print('📁 Modèle ONNX sauvegardé sur Drive : agario_rl_backup_v3/model_v3.onnx')
else:
    print('⚠️ Aucun checkpoint trouvé pour l\'export ONNX.')


## 7. Visionner & Rejouer Directement le Modèle (Google Drive ou Local)
Cette cellule charge votre meilleur modèle (PyTorch `.zip` ou ONNX `.onnx`) depuis Drive (`agario_rl_backup_v3` ou `agario_rl_backup_v2`), simule un match de 80 secondes avec la physique exacte, et affiche la vidéo HD directement ci-dessous.


In [ ]:
import os, sys, glob, re
from IPython.display import HTML, display
from base64 import b64encode

def extract_step(path):
    if 'final' in os.path.basename(path):
        return 999999999
    m = re.search(r'step_(\d+)', path)
    return int(m.group(1)) if m else 0

# Recherche automatique du meilleur modèle disponible
search_paths = [
    '/content/drive/MyDrive/agario_rl_backup_v3',
    '/content/drive/MyDrive/agario_rl_backup_v2',
    'models',
    'checkpoints/ppo'
]
all_ckpts = []
for p in search_paths:
    if os.path.exists(p):
        all_ckpts.extend(glob.glob(os.path.join(p, '*.zip')))
        all_ckpts.extend(glob.glob(os.path.join(p, '*.onnx')))

all_ckpts = sorted(list(set(all_ckpts)), key=extract_step, reverse=True)
selected_model = all_ckpts[0] if all_ckpts else None

if not selected_model:
    print('⚠️ Aucun modèle trouvé. Spécifiez un chemin direct vers votre checkpoint.')
else:
    print('=' * 75)
    print(f'⚡ Modèle sélectionné : {selected_model}')
    print('=' * 75)
    os.makedirs('recordings', exist_ok=True)
    video_path = 'recordings/eval_replay.mp4'
    !python src/inference/record_match.py --model "{selected_model}" --output "{video_path}" --steps 2400
    if os.path.exists(video_path):
        mp4_bytes = open(video_path, 'rb').read()
        data_url = 'data:video/mp4;base64,' + b64encode(mp4_bytes).decode()
        display(HTML(f'''<div style="text-align:center; margin:15px 0;">
            <h3 style="color:#2c3e50;">🎮 Match Replay HD (Multi-Agent Simulation)</h3>
            <video width="850" height="480" controls autoplay loop style="border-radius:8px; box-shadow:0 4px 12px rgba(0,0,0,0.25);">
                <source src="{data_url}" type="video/mp4">
            </video>
        </div>'''))
